In [18]:
import pandas as pd
import sys

Для нерекурсивного решения была взята система непересекающихся множеств

Алгоритм Union-Find 

In [19]:
class UnionFind:
    def __init__(self):
        self.parent = {}
    
    def add(self, x):
        if x not in self.parent:
            self.parent[x] = x
    
    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    
    def union(self, x, y):
        self.add(x)
        self.add(y)
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            if rx < ry:
                self.parent[ry] = rx
            else:
                self.parent[rx] = ry

In [20]:
def resolve_ids(users_df, links_df):
    uf = UnionFind()
    
    for _, row in users_df.iterrows():
        uf.add(row['id'])
        uf.add(row['new_id'])
    
    for _, row in links_df.iterrows():
        uf.add(row['id1'])
        uf.add(row['id2'])
    
    for _, row in users_df.iterrows():
        uf.union(row['id'], row['new_id'])
    
    for _, row in links_df.iterrows():
        uf.union(row['id1'], row['id2'])
    
    result = []
    for node in uf.parent:
        result.append({'id': node, 'new_id': uf.find(node)})
    
    return pd.DataFrame(result).sort_values('id').reset_index(drop=True)

Для рекурсивного решения был выбран стандартный алгоритм DFS с рекурсией
из‑за простоты реализации и наглядности


In [24]:
def resolve_ids_dfs(users_df, links_df):
    all_ids = set(users_df['id']).union(set(users_df['new_id'])) \
               .union(set(links_df['id1'])).union(set(links_df['id2']))

    all_ids.discard(0)
    
    graph = {uid: set() for uid in all_ids}
    
    for _, row in users_df.iterrows():
        a, b = row['id'], row['new_id']
        if a != 0 and b != 0:   
            graph[a].add(b)
            graph[b].add(a)
    
    for _, row in links_df.iterrows():
        a, b = row['id1'], row['id2']
        if a != 0 and b != 0:
            graph[a].add(b)
            graph[b].add(a)
    
    visited = set()
    components = []
    
    def dfs(node, comp):
        visited.add(node)
        comp.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor, comp)
    
    for node in graph:
        if node not in visited:
            comp = []
            dfs(node, comp)
            components.append(comp)
    
    result = {}
    for comp in components:
        min_id = min(comp)
        for node in comp:
            result[node] = min_id
    
    return pd.DataFrame(result.items(), columns=['id', 'new_id'])

In [22]:
# Пример
users_data = {
    'id': [1, 2, 3, 4, 5, 7, 8],
    'new_id': [2, 3, 0, 4, 0, 8, 0]
    }  

users_df = pd.DataFrame(users_data)

users_df = users_df[users_df['new_id'] != 0]
    
links_data = {
    'id1': [3, 7],
    'id2': [5, 8]
    }
links_df = pd.DataFrame(links_data)
    
result = resolve_ids(users_df, links_df)

result

,id,new_id
0,1,1
1,2,1
2,3,1
3,4,4
4,5,1
5,7,7
6,8,7


In [23]:
result = resolve_ids_dfs(users_df, links_df)
result

,id,new_id
0,1,1
1,2,1
2,3,1
3,5,1
4,4,4
5,7,7
6,8,7
